<a href="https://colab.research.google.com/github/NikoriakViktor/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_3/bonus/pandas_data_analysis/note_bonus_extra_datasets_student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧪 Бонус М3 — додаткові датасети: реальні дані, реальні проблеми

Перед зустріччю з Мартою Петрівною Ліна ставить собі незручні питання. «Олія ×6» — це багато чи мало, якщо зарплати теж росли? Чи справді 2022 рік — злам? Чи дешева їжа в Україні порівняно зі світом? Волонтер Тарас приносить нові файли — і кожен зі своєю «хворобою». У цьому ноутбуці ми лікуємо їх по черзі й після **кожного** кроку дивимось, що змінилось у таблиці (`shape`, `head`).

| Розділ | Датасет | Що всередині | Приклади | Задача |
|---|---|---|---|---|
| 0 | — | підготовка, функція `look` для огляду таблиці | 0.1 | |
| 1 | зарплати Держстату, порти | `read_csv`: `encoding="cp1251"`, `sep`, `header`, `thousands`, `na_values`, `nrows`, `decimal`, `parse_dates` | 1.1–1.6 | |
| 2 | ціни WFP в Україні | огляд: `info`, `describe`, `value_counts`, `isna`, `nunique`, змішані одиниці | 2.1–2.5 | |
| 3 | зарплати Держстату (CSV) | «брудний» файл крок за кроком з перевіркою після кожного кроку → `melt` | 3.1–3.7 | **Задача 1** (той самий файл у xlsx) |
| 4 | зарплати, ціни | фільтри: булеві маски, `&` `\|` `~`, `isin`, `between`, `query` і `@змінна`, копія після фільтра | 4.1–4.6 | |
| 5 | зарплати, ціни | Split-Apply-Combine: `groupby`, `agg`, `transform`, `rank` | 5.1–5.5 | |
| 6 | зарплата проти хліба | `merge` і пастки: тип ключа, ключі без пари (`indicator`), «багато-до-багатьох» (`validate`) | 6.1–6.6 | **Задача 2** (хліб на зарплату по областях) |
| 7 | Світовий банк, ціни | `pivot`, `pivot_table`, `melt` назад | 7.1–7.4 | |
| 8 | порти, щоденно 2019–2026 | час: часовий пояс, `sort_index`, `resample`, зрізи за датами, «до/після» | 8.1–8.7 | **Задача 3** (порти до і після 24.02.2022) |
| 9 | ціни WFP у 71 країні | великий файл: `usecols`, медіана, рейтинг | 9.1–9.2 | |

**Як працювати**

- Виконуй клітинки **зверху вниз**. Клітинка з визначенням (`def`, словник-довідник, константи) лише створює об'єкт — **запуск завжди в наступній клітинці**.
- Після кожного перетворення дивись на `shape` і перші рядки: так помилка видно одразу, а не за три кроки.
- Перед **🔮 Прогнозом** спершу відповідай сам, потім розгортай відповідь.
- Код працює і в pandas 2.x (Colab), і в pandas 3.0. Де вивід відрізняється (тип рядків `str` у pandas 3 проти `object` у pandas 2), це сказано в тексті.
- Теорія — у книзі курсу: [Бонус. Pandas](https://nikoriakviktor.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m3/bonus_pandas/) і [довідник з аналізу даних](https://nikoriakviktor.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m3/pandas/data_analytics/).

---
## 0. Підготовка

У Colab перша клітинка завантажує папку уроку разом із даними й доставляє бібліотеки, яких бракує (`openpyxl` потрібен для `.xlsx`).

In [ ]:
import importlib.util, io, os, subprocess, sys

LESSON = "module_3/bonus/pandas_data_analysis"
REPO = "https://github.com/NikoriakViktor/PY-Course-Victor-Nikoriak-22-09-2026.git"
if not os.path.isdir("data"):
    if not os.path.isdir("course_repo"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--filter=blob:none", "--sparse", REPO, "course_repo"], check=True)
        subprocess.run(["git", "-C", "course_repo", "sparse-checkout", "set", LESSON], check=True)
    os.chdir(os.path.join("course_repo", LESSON))
missing = [m for m in ("pandas", "matplotlib", "openpyxl") if importlib.util.find_spec(m) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

DATA_DIR = "data"
SALARY = os.path.join(DATA_DIR, "101-serednomisiachna-nominalna-zarobitna-plata-za-regionami-u-rozrakhunku-na-odnogo-shtatnogo-pr.csv")
SALARY_XLSX = os.path.join(DATA_DIR, "101-serednomisiachna-nominalna-zarobitna-plata-za-regionami-u-rozrakhunku-na-odnogo-shtatnogo-p.xlsx")
FOOD = os.path.join(DATA_DIR, "wfp_food_prices_ukr.csv")
MARKETS = os.path.join(DATA_DIR, "wfp_markets_ukr.csv")
WORLD_BANK = os.path.join(DATA_DIR, "climate-change_ukr.csv")
PORTS = os.path.join(DATA_DIR, "ukraine-daily-port-activity-data-and-shipment-estimates.csv")
WORLD = os.path.join(DATA_DIR, "wfp_food_prices_global_2026.csv")
print("файлів у data/:", len(os.listdir(DATA_DIR)), "| pandas", pd.__version__)

### Приклад 0.1. Функція `look`: розмір і перші рядки одним викликом

Ми весь час питатимемо таблицю «скільки тебе і як ти виглядаєш». Щоб не писати `print(df.shape)` і `df.head()` щоразу, оголосимо маленьку функцію. Запуск — у наступній клітинці.

In [ ]:
def look(df, title="", n=3):
    """Показати назву кроку, розмір (рядки, стовпці) і перші n рядків таблиці."""
    print(f"{title}: {df.shape[0]:,} рядків × {df.shape[1]} стовпців")
    display(df.head(n))

In [ ]:
demo = pd.DataFrame({"місто": ["Київ", "Львів", "Одеса", "Харків"], "ціна": [34.0, 32.5, 33.0, 31.0]})
look(demo, "демо", n=2)

Функція друкує рядок «назва: рядки × стовпці» і показує перші `n` рядків. Далі після кожного перетворення — `look(...)`.

---
## 1. Читання файлів: параметри `read_csv`

Тарас приносить файл Держстату із зарплатами за регіонами (data.gov.ua). Перш ніж читати його pandas, подивимось на **сирі байти** — так видно кодування й роздільники.

### Приклад 1.1. Сирі байти: яке це кодування?

In [ ]:
with open(SALARY, "rb") as f:
    head_bytes = f.read(120)
print(head_bytes)

Байти `\xca\xe2…` — це **cp1251** (Windows-кодування кирилиці), не UTF-8. У UTF-8 кожна кирилична літера займає 2 байти й починається з `\xd0` або `\xd1`. Перевіримо: декодуємо ті самі байти обома способами.

In [ ]:
print("cp1251:", head_bytes.decode("cp1251")[:70])
print("utf-8 :", head_bytes.decode("utf-8", errors="replace")[:70])

У cp1251 — нормальний текст `; Показники,2000,2005…`, у UTF-8 — «кракозябри» `�`. Ще видно: рядок починається з `;`, усередині — коми, тисячі відділені **пробілом** (`1 041`), а замість чисел трапляються `…` (немає даних) і `х` (показник не рахувався).

### Приклад 1.2. Що буде, якщо читати «як завжди»

🔮 **Прогноз:** `pd.read_csv(SALARY)` без параметрів — що станеться?

<details>
<summary>Відповідь</summary>

Помилка `UnicodeDecodeError`: pandas за замовчуванням читає UTF-8, а байт `\xa0` (нерозривний пробіл у cp1251) не є початком жодного символу UTF-8.

</details>

In [ ]:
attempts = {
    "без параметрів": {},
    "encoding='cp1251'": {"encoding": "cp1251"},
}

In [ ]:
for label, options in attempts.items():
    try:
        pd.read_csv(SALARY, **options)
    except Exception as error:
        print(f"{label:20} → {type(error).__name__}: {str(error)[:90]}")

Перша спроба падає на кодуванні. Друга вже декодує текст, але падає на структурі: `ParserError … Expected 16 fields in line 32, saw 17` — у кінці файлу є примітки, де ком більше, ніж стовпців. Тобто файл — не зовсім таблиця: зверху заголовок, знизу текст.

### Приклад 1.3. Параметри `read_csv` роблять більшу частину роботи

Спробуємо описати файл параметрами:

- `sep=","` — роздільник усередині рядка;
- `thousands=" "` — пробіл між тисячами (`1 041` → `1041`);
- `na_values=["…", "х"]` — ці позначки читати як `NaN`;
- `nrows=29` — лише 29 рядків даних, примітки внизу не читаємо.

In [ ]:
by_options = pd.read_csv(SALARY, encoding="cp1251", sep=",", thousands=" ", na_values=["…", "х"], nrows=29)
look(by_options, "read_csv з параметрами")
print(by_options.dtypes.value_counts())
print("перший стовпець:", repr(by_options.columns[0]), "| останній:", repr(by_options.columns[-1]))

Майже вийшло: 29 рядків × 16 стовпців, числа в середині вже числові (`float64` там, де є `NaN`, `int64` — де пропусків немає). Але два крайні стовпці зіпсовані тими самими `;`: перший називається `';\xa0Показники'`, а регіони мають вигляд `;Україна`; останній — `'2018;;;;…'`, і значення в ньому лишились **рядками** (`8 865;;;;`), бо `;` заважає розпізнати число. Тип такого стовпця — `str` у pandas 3 і `object` у pandas 2.

Правило: параметри `read_csv` — перший інструмент очищення, але після читання **завжди** перевіряй `dtypes` і назви стовпців. Нижче, у розділі 3, ми розберемо цей файл надійніше — вручну, з перевіркою на кожному кроці.

### Приклад 1.4. `decimal` і `sep=";"`: «український Excel»

Excel з українськими налаштуваннями зберігає CSV із `;` як роздільником і **комою** в дробових числах (`34,50`), а дати пише як `15.03.2024`. Такий файл Тарас надсилає щотижня. Створимо його текстом і прочитаємо з `io.StringIO` (це «файл у пам'яті»).

In [ ]:
TARAS_CSV = """дата;ринок;ціна
15.03.2024;Київ;34,50
15.03.2024;Львів;32,00
15.04.2024;Київ;35,10
15.04.2024;Львів;32,75
"""

In [ ]:
naive = pd.read_csv(io.StringIO(TARAS_CSV))
look(naive, "без параметрів", n=2)

prices = pd.read_csv(io.StringIO(TARAS_CSV), sep=";", decimal=",", parse_dates=["дата"], date_format="%d.%m.%Y")
look(prices, "sep=';', decimal=',', parse_dates")
print(prices.dtypes.astype(str).to_dict())
print("середня ціна:", prices["ціна"].mean())

Без параметрів pandas шукає **кому** — і знаходить її лише в ціні `34,50`. Тому вийшов один стовпець з назвою `дата;ринок;ціна`, а шматок рядка до коми (`15.03.2024;Київ;34`) потрапив в **індекс**: таблиця «прочиталась», але це сміття. З `sep=";"` — три стовпці; `decimal=","` перетворив `34,50` на число 34.5; `parse_dates` з `date_format="%d.%m.%Y"` прочитав дати в правильному порядку «день.місяць.рік» (тип `datetime64`). Тепер середнє рахується: 33.5875. Тип `ринок` — `str` у pandas 3 і `object` у pandas 2.

### Приклад 1.5. Без `date_format` дата лишається текстом або читається не так

In [ ]:
as_text = pd.read_csv(io.StringIO(TARAS_CSV), sep=";", decimal=",")
print("без parse_dates:", as_text["дата"].dtype, "| перше значення:", repr(as_text["дата"].iloc[0]))
print("місяць з pd.to_datetime(dayfirst=True):", pd.to_datetime(as_text["дата"], dayfirst=True).dt.month.tolist())

Без `parse_dates` дата — просто рядок `'15.03.2024'`: ні сортування за часом, ні `.dt.month`, ні `resample`. Якщо файл уже прочитано, дату можна виправити окремо: `pd.to_datetime(..., dayfirst=True)` дає місяці `[3, 3, 4, 4]`. Формат краще задавати явно: «03.04.2024» без підказки легко прочитати як 4 березня.

### Приклад 1.6. `parse_dates` і часовий пояс у файлі портів

У файлі портів дати записані як `2026-03-27 00:00:00+00:00` — з часовим поясом UTC.

In [ ]:
ports = pd.read_csv(PORTS, parse_dates=["date"])
look(ports[["date", "portname", "portcalls", "export"]], "порти")
print("тип date:", ports["date"].dtype)

`parse_dates` перетворив рядки на `datetime64[…, UTC]` — дати **з часовим поясом** (`ns` у pandas 2, `us` у pandas 3 — це лише точність зберігання). Запам'ятай `UTC`: у розділі 8 саме він дасть пастку при порівнянні дат. І ще деталь: перший рядок — 2026 рік, тобто файл відсортований **від нових до старих**.

---
## 2. Огляд даних: перше знайомство

Перш ніж рахувати, Ліна оглядає таблицю цін WFP — так, як детектив оглядає місце події: скільки рядків, які стовпці, де пропуски, які значення трапляються.

### Приклад 2.1. `shape`, `head` і `info`

In [ ]:
food = pd.read_csv(FOOD, parse_dates=["date"])
look(food, "ціни WFP")

In [ ]:
food.info()

`info()` показує кількість рядків (112 259), для кожного стовпця — скільки **непорожніх** значень і тип. Видно одразу: `admin1`, `admin2`, `latitude`, `longitude` заповнені не всюди (111 273 з 112 259). Текстові стовпці мають тип `str` у pandas 3 або `object` у pandas 2 — зміст той самий. Останній рядок — пам'ять, яку займає таблиця.

### Приклад 2.2. `describe`: швидкий «рентген» чисел

In [ ]:
print(food[["price", "usdprice"]].describe().round(2))

Мінімум, квартилі, медіана (`50%`), максимум. Середнє (48,85 грн) майже вдвічі більше за медіану (28,10 грн), а максимум — 434 грн. Це не помилка: у таблиці разом лежать хліб, м'ясо, ліки й пальне в різних одиницях, і кілька дорогих товарів «тягнуть» середнє вгору. Тому наступний крок — подивитись на категорії.

### Приклад 2.3. `value_counts`: які значення трапляються

In [ ]:
print(food["unit"].value_counts())
print()
print(food["admin1"].value_counts(dropna=False).tail(4))

Одиниць дев'ять: кілограми, літри, «10 таблеток», «10 штук», квиток, `Loaf` (буханка)… Середнє по всій таблиці змішує їх — тому `describe()` вище нічого не каже про «ціну їжі». У `admin1` з `dropna=False` видно 986 рядків `NaN` — це ринок «National Average» (ціна по країні), а Крим і Севастополь мають лише 21 і 43 рядки: дані звідти обриваються у 2014 році.

### Приклад 2.4. `isna`, `nunique`, `duplicated`: пропуски, різноманіття, дублікати

In [ ]:
print("пропуски по стовпцях:")
print(food.isna().sum()[lambda s: s > 0])
print("\nунікальних:", food[["market", "commodity", "unit", "currency", "pricetype"]].nunique().to_dict())
print("повних дублікатів:", food.duplicated().sum(),
      "| дублікатів за (date, market, commodity):", food.duplicated(["date", "market", "commodity"]).sum())

Пропуски — лише в «географічних» стовпцях (у National Average немає області й координат). Ринків 28 (27 міст + National Average), товарів 45, валюта й тип ціни — по одному значенню (`UAH`, `Retail`): ці стовпці можна не аналізувати. Дублікатів немає: одна ціна на (дату, ринок, товар). Це стане в пригоді в розділі 7.

### Приклад 2.5. Змішані одиниці: той самий товар — у різних одиницях

Ліна хоче ряд цін на хліб. Подивимось, які одиниці й роки є для кожного виду хліба.

In [ ]:
bread_units = (
    food[food["commodity"].str.startswith("Bread")]
    .groupby(["commodity", "unit"])["date"]
    .agg(["min", "max", "count"])
)
print(bread_units)

`Bread (rye)` (житній) є у двох одиницях: `Loaf` (буханка) у 2014–2017 і `KG` з грудня 2018; `Bread (wheat)` — пакування `500 G`. Склеїти такі рядки в один графік — означає порівнювати буханку з кілограмом. Правило: **перед будь-яким середнім перевір `unit`**.

---
## 3. Очищення «брудного» файлу крок за кроком

Повертаємось до зарплат. План:

```
сирий файл ─► один стовпець між «;» ─► розбити по комі ─► заголовок + рядки даних
          ─► «широка» таблиця ─► прибрати пробіли в назвах ─► melt у «довгу» ─► числа ─► перевірка
```

Після кожного кроку — перевірка: що змінилось і чи це те, що ми хотіли.

### Приклад 3.1. Прочитати файл як є: `sep=";"`, `header=None`

Кожен рядок файлу обгорнутий у `;…;;;;`. Якщо роздільником вважати `;`, то **весь** вміст рядка опиниться в одному стовпці з індексом 1. `header=None` — не вважати перший рядок заголовком (він зламаний).

In [ ]:
raw = pd.read_csv(SALARY, encoding="cp1251", sep=";", header=None)
look(raw.iloc[:, :3], "сирий файл, перші 3 стовпці")
print("стовпців:", raw.shape[1], "| порожніх значень у стовпці 1:", raw[1].isna().sum())

33 рядки й 24 стовпці: стовпець 0 порожній (до першого `;`), стовпець 1 — увесь текст рядка, решта — порожні «хвости» `;;;;`. Порожнє значення в стовпці 1 — одне (останній рядок файлу складається лише з `;`), тож рядків з текстом — 32: заголовок, 29 рядків даних і 2 примітки.

### Приклад 3.2. Розбити кожен рядок по комі

In [ ]:
rows = raw[1].dropna().str.split(",")
print("рядків:", len(rows))
print(rows.iloc[0])
print(rows.iloc[2])
print("довжини списків:", rows.map(len).value_counts().to_dict())

`str.split(",")` перетворив кожен рядок на **список**. Перший список — заголовок (`'\xa0Показники'` і роки 2000–2018), третій — `Україна*` з `х` за 2000–2009. Довжини: 31 список по 16 елементів і один — 17. Але 31 = заголовок + 29 рядків даних + **одна примітка**, у якій випадково теж 15 ком (`…10 і більше осіб. ,,,,`). Тому лише довжини мало — потрібна ще одна умова.

### Приклад 3.3. Зібрати «широку» таблицю: лише повні рядки

Залишаємо рядки, у яких стільки ж елементів, скільки в заголовку, **і** непорожнє перше число (у примітки на його місці — порожній рядок `''`). Перший стовпець назвемо `регіон`, решта — роки.

In [ ]:
header = rows.iloc[0]
data = [row for row in rows.iloc[1:] if len(row) == len(header) and row[1] != ""]
wide = pd.DataFrame(data, columns=["регіон"] + header[1:])
look(wide, "широка таблиця")

29 рядків × 16 стовпців: 24 області, Крим, Київ, Севастополь і два рядки «Україна». Але **всі** значення ще текстові — навіть `230`.

### Приклад 3.4. Невидимі пробіли в назвах

У заголовку перед «Показники» стоїть нерозривний пробіл `\xa0` — на екрані його не видно. Перевіримо, чи є такі пробіли в назвах регіонів, і приберемо їх.

In [ ]:
print("заголовок першого стовпця:", repr(header[0]))
padded = wide["регіон"] != wide["регіон"].str.strip()
print("назв регіонів з пробілами по краях:", padded.sum())
wide["регіон"] = wide["регіон"].str.strip()
print("Київ знаходиться:", (wide["регіон"] == "м. Київ").sum())

У заголовку `\xa0` є, а в назвах регіонів цього файлу — ні (0). `.str.strip()` все одно лишаємо: він дешевий, прибирає і звичайні, і нерозривні пробіли, а в інших версіях таблиці чи в іншому джерелі такий пробіл легко трапляється — і тоді `== "м. Київ"` не спрацює, а `merge` з іншою таблицею тихо втратить рядок. Перевірка після кроку: Київ знаходиться рівно один раз.

### Приклад 3.5. `melt`: з «широкої» таблиці — у «довгу»

«Широка» таблиця зручна для очей (рік — стовпець), але для групування й графіків потрібна «довга»: один рядок = (регіон, рік, значення).

In [ ]:
salary = wide.melt(id_vars="регіон", var_name="рік", value_name="зарплата")
look(salary, "після melt")
print("очікуємо 29 × 15 =", 29 * 15)

`melt` «розплавив» 15 стовпців-років у два стовпці: `рік` (назва колишнього стовпця) і `зарплата` (значення). Рядків стало 435 = 29 регіонів × 15 років.

```
широка:  регіон | 2000 | 2005 | …          довга:  регіон  | рік  | зарплата
         Україна|  230 |  806 | …    ─►            Україна | 2000 | 230
                                                   Україна | 2005 | 806
```

### Приклад 3.6. Числа: пробіли, `…` і `х`

In [ ]:
print("приклади значень:", salary["зарплата"].iloc[[0, 30, 100, 350]].tolist())
salary["зарплата"] = pd.to_numeric(salary["зарплата"].str.replace(" ", ""), errors="coerce")
salary["рік"] = salary["рік"].astype(int)
print(salary.dtypes.astype(str).to_dict())
print("пропусків:", salary["зарплата"].isna().sum())

Спершу прибираємо пробіл-роздільник тисяч (`'1 041'` → `'1041'`), потім `pd.to_numeric(..., errors="coerce")`: усе, що не є числом (`…`, `х`), стає `NaN`, а не ламає код. `рік` — з тексту `'2000'` у ціле число. Тепер `зарплата` — `float64` (через `NaN` — не `int`), `рік` — `int64`, `регіон` лишається текстом (`str` у pandas 3, `object` у pandas 2). Пропусків 21.

### Приклад 3.7. Перевірка результату: звідки 21 пропуск

In [ ]:
gaps = salary[salary["зарплата"].isna()]
print(gaps.groupby("регіон")["рік"].agg(["min", "max", "count"]))
assert salary["рік"].between(2000, 2018).all()
assert salary.loc[(salary["регіон"] == "Україна*") & (salary["рік"] == 2018), "зарплата"].item() == 8865
print("✔ роки 2000–2018, Україна* 2018 = 8865 грн")

Пропуски пояснювані: `Україна`, Крим і Севастополь — 2014–2018 (`…`, дані з 2014 року без тимчасово окупованих територій), `Україна*` — 2000–2009 (`х`, ряд рахують з 2010 року). Разом 5 + 5 + 5 + 6 = 21. Пропуск, який можна пояснити, — це інформація; пропуск, який не можна, — привід шукати помилку в очищенні.

Два `assert` — «запобіжники»: якщо хтось змінить кроки вище, ноутбук упаде тут, а не видасть неправильний висновок далі.

**🔮 Прогноз:** У скільки разів зарплата в Києві 2018 року більша за найнижчу серед областей?

<details>
<summary>Відповідь</summary>

Порахуй у наступній клітинці. Київ — 13 542 грн, найнижча — Тернопільська, 6 969 грн: у 1,94 раза.

</details>

In [ ]:
year2018 = salary.query("рік == 2018").dropna(subset=["зарплата"])
print(year2018.nlargest(3, "зарплата").to_string(index=False))
print(year2018.nsmallest(2, "зарплата").to_string(index=False))
lowest = year2018.nsmallest(1, "зарплата").iloc[0]
print(f"Київ / {lowest['регіон']}: {year2018['зарплата'].max() / lowest['зарплата']:.2f}")

`nlargest` / `nsmallest` — найбільші й найменші рядки за стовпцем без повного сортування. Київ — 13 542 грн, далі Донецька й Київська; найнижча — Тернопільська (6 969 грн). Різниця — 1,94 раза.

### Задача 1. Той самий файл у форматі Excel

Поруч лежить `.xlsx`-версія таблиці (`SALARY_XLSX`) — і в ній на три роки більше: до 2021-го. Excel уже зберігає числа як числа, але `…` і `х` там лишились, а внизу — ті самі примітки.

Збери «довгу» таблицю `salary_x` зі стовпцями `регіон`, `рік`, `зарплата`, як у прикладах 3.3–3.6:

1. `pd.read_excel(SALARY_XLSX)` — подивись на результат через `look`: роки вже стали назвами стовпців, перший стовпець називається `Unnamed: 0`;
2. перейменуй перший стовпець на `регіон` і відкинь рядки-примітки (у них порожньо в стовпці `2000`);
3. `melt`, потім `pd.to_numeric(..., errors="coerce")` для `зарплата` і `int` для `рік`.

Перевірка — у наступній клітинці: вона порівняє твою таблицю з `salary` за 2000–2018.

In [ ]:
# YOUR CODE HERE


In [ ]:
assert "salary_x" in globals(), "створи salary_x у клітинці задачі вище"
assert list(salary_x.columns) == ["регіон", "рік", "зарплата"], "стовпці мають бути: регіон, рік, зарплата"
assert len(salary_x) == 29 * 18, f"очікуємо 29 регіонів × 18 років = 522 рядки, а є {len(salary_x)}: чи відкинуто примітки?"
assert salary_x["рік"].min() == 2000 and salary_x["рік"].max() == 2021
assert pd.api.types.is_numeric_dtype(salary_x["зарплата"]), "зарплата має бути числом: pd.to_numeric(..., errors='coerce')"
both = salary.merge(salary_x, on=["регіон", "рік"], suffixes=("_csv", "_xlsx"))
assert len(both) == len(salary), "назви регіонів у двох файлах мають збігтися"
same = both.dropna(subset=["зарплата_csv"])
assert (same["зарплата_csv"] == same["зарплата_xlsx"]).all(), "за 2000–2018 xlsx має давати ті самі числа, що й CSV"
top2021 = salary_x.query("рік == 2021").nlargest(3, "зарплата")
print(top2021.to_string(index=False))
print("✅ Задача 1 розв'язана")

---
## 4. Фільтри: булеві маски і `query`

Фільтр у pandas — це **маска**: Series з `True`/`False` тієї ж довжини, що й таблиця. `df[маска]` лишає рядки з `True`.

### Приклад 4.1. Одна умова — одна маска

In [ ]:
mask_kyiv = salary["регіон"] == "м. Київ"
print(mask_kyiv.head(3).tolist(), "… True у масці:", mask_kyiv.sum())
kyiv = salary[mask_kyiv]
look(kyiv, "лише Київ")

Маска — 435 значень `True`/`False`; `True` — 15 (Київ за 15 років). `salary[mask_kyiv]` — 15 рядків, індекс зберігся з початкової таблиці (27, 56, …): pandas не перенумеровує рядки після фільтра.

### Приклад 4.2. Кілька умов: `&`, `|`, `~` і дужки

In [ ]:
recent_big = salary[(salary["рік"] >= 2016) & (salary["зарплата"] > 8000)]
look(recent_big, "з 2016 і > 8000 грн", n=5)

`&` — «і», `|` — «або», `~` — «не». Кожна умова — **у дужках**. Таких рядків 13: Київ — у 2016 і 2017 роках (єдиний), а у 2018-му до нього приєднались ще 9 областей і рядок `Україна*`.

🔮 **Прогноз:** що буде без дужок: `salary["рік"] >= 2016 & salary["зарплата"] > 8000`?

<details>
<summary>Відповідь</summary>

Помилка. `&` у Python виконується **раніше** за `>=` і `>`, тож pandas спершу рахує `2016 & salary["зарплата"]` — побітове «і» між числом і дробовими зарплатами, якого не існує.

</details>

In [ ]:
try:
    salary[salary["рік"] >= 2016 & salary["зарплата"] > 8000]
except TypeError as error:
    print(type(error).__name__, "→", str(error)[:80])

`TypeError: Cannot perform 'rand_' …` — `rand_` і є оператор `&`, застосований до `float64` і числа 2016. Повідомлення незрозуміле, бо помилка не там, де ми думаємо. Правило: умови в масках — завжди в дужках.

### Приклад 4.3. `isin`, `between`, `~` і `.str`

In [ ]:
not_regions = salary["регіон"].str.startswith("Україна") | salary["регіон"].isin(["Автономна Республіка Крим", "м. Севастополь"])
regions = salary[~not_regions & salary["рік"].between(2010, 2018)]
look(regions, "області + Київ, 2010–2018")
print("регіонів:", regions["регіон"].nunique(), "| пропусків:", regions["зарплата"].isna().sum())

- `.str.startswith("Україна")` — рядки «Україна» й «Україна*»;
- `.isin([...])` — значення з переліку;
- `~` — «не»: залишаємо все, що **не** підсумок і не Крим/Севастополь;
- `.between(2010, 2018)` — включно з обома межами.

Результат: 25 регіонів (24 області + Київ) × 9 років = 225 рядків, і жодного пропуску. Таблицю `regions` використаємо в розділі 5.

### Приклад 4.4. `query`: та сама логіка текстом

In [ ]:
by_mask = food[(food["commodity"] == "Sugar") & (food["market"] == "Kyiv city") & (food["date"] >= "2026-01-01")]
by_query = food.query("commodity == 'Sugar' and market == 'Kyiv city' and date >= '2026-01-01'")
look(by_query[["date", "market", "commodity", "unit", "price"]], "цукор у Києві, 2026", n=8)
print("однаково:", by_mask.equals(by_query))

`query` — рядок з умовами: `and`/`or`/`not` замість `&`/`|`/`~`, без повторення `food[...]` і без дужок навколо кожної умови. Результат той самий (`equals` → `True`): 8 місяців 2026 року, ціна цукру в Києві — від 29 з лишком до майже 33 грн/кг.

### Приклад 4.5. `query` зі змінною: `@`

In [ ]:
basket = ["Sugar", "Potatoes", "Oil (sunflower)"]
limit = 50
day = pd.Timestamp("2026-08-15")
cheap = food.query("commodity in @basket and price < @limit and admin1.notna()")
look(cheap[["date", "market", "commodity", "unit", "price"]], "кошик дешевше 50 грн")
print(cheap["commodity"].value_counts().to_dict())
print("рядків за", day.date(), ":", len(food.query("date == @day")))

`@basket`, `@limit`, `@day` — змінні Python усередині рядка `query`. `admin1.notna()` — методи стовпців теж працюють. Олії дешевше 50 грн/л знайшлось майже вдвічі менше, ніж цукру й картоплі: з 2022 року вона здебільшого дорожча.

Дату для **рівності** передаємо як `pd.Timestamp` через `@`. Рядок `"date == '2026-08-15'"` pandas 2 ще зіставляє з датою (з попередженням), а pandas 3 — вже ні: результат буде порожнім без жодної помилки. Порівняння `>=` з рядком (як у прикладі 4.4) працює в обох версіях.

### Приклад 4.6. Фільтр — це нова таблиця: змінюй її через `assign`

Ліна хоче додати до київських рядків стовпець «тисяч грн». Якщо після фільтра написати `kyiv["тис"] = ...`, pandas 2 покаже `SettingWithCopyWarning` («ти змінюєш копію чи оригінал?»). У pandas 3 діє **Copy-on-Write**: результат фільтра — завжди окрема таблиця, попередження немає. Код, однаковий для обох версій, — `assign` (повертає нову таблицю) або `.copy()` після фільтра.

In [ ]:
kyiv = salary[salary["регіон"] == "м. Київ"].assign(тис_грн=lambda d: (d["зарплата"] / 1000).round(1))
look(kyiv.tail(3), "Київ з новим стовпцем")
print("у salary з'явився стовпець?", "тис_грн" in salary.columns)

Новий стовпець є лише в `kyiv`, оригінал `salary` не змінився. Якщо ж треба змінити **оригінал** за умовою — один виклик `.loc[маска, "стовпець"] = значення`, а не ланцюжок `df[...][...] = ...` (у pandas 3 ланцюжок мовчки нічого не змінить).

---
## 5. Split-Apply-Combine: `groupby`, `agg`, `transform`

Головна операція аналітика: **розбити** таблицю на групи (split), **застосувати** функцію до кожної (apply), **зібрати** результат (combine).

```
regions (225 рядків) ─ groupby("рік") ─► 9 груп по 25 регіонів ─ mean() ─► 9 чисел ─► Series з індексом «рік»
```

### Приклад 5.1. Одна агрегація: середня зарплата по роках

In [ ]:
mean_by_year = regions.groupby("рік")["зарплата"].mean().round(0)
print(mean_by_year)

Із 225 рядків — 9 значень, по одному на рік; індекс — `рік`. Це **просте** середнє по регіонах: Київ і Чернівецька важать однаково, тому воно відрізняється від рядка `Україна*` (там Держстат зважує на кількість працівників).

### Приклад 5.2. Кілька агрегацій з назвами: `agg`

In [ ]:
stats = regions.groupby("рік").agg(
    регіонів=("регіон", "nunique"),
    мінімум=("зарплата", "min"),
    медіана=("зарплата", "median"),
    максимум=("зарплата", "max"),
)
stats["розрив_разів"] = (stats["максимум"] / stats["мінімум"]).round(2)
look(stats, "статистика по роках", n=9)

«Іменована агрегація»: `нова_назва=("стовпець", "функція")`. Найбільша зарплата щороку — у Києві. Розрив між найбагатшим і найбіднішим регіоном тримається біля 2 разів: зростав до 2,34 у 2016 році й повернувся до 1,94 у 2018-му.

### Приклад 5.3. Групування за двома ключами на цінах

In [ ]:
sugar_regional = food.query("commodity == 'Sugar' and admin1.notna()").assign(рік=lambda d: d["date"].dt.year)
sugar_by = sugar_regional.groupby(["рік", "market"], as_index=False)["price"].mean()
look(sugar_by, "цукор: рік × ринок")
print(sugar_by.groupby("рік")["market"].nunique().to_dict())

`as_index=False` — ключі групування лишаються **стовпцями**, а не індексом; результат одразу придатний для `merge` чи графіка. Друга група показує, скільки ринків мало дані щороку: 27 у 2014-му, 25 у 2015–2022, 23 з 2023 року. «Середня ціна по Україні» в різні роки рахується по різних ринках — пам'ятай про це, коли порівнюєш роки.

### Приклад 5.4. `transform`: результат групи — у кожен рядок

Ліна хоче для кожного регіону знати, у скільки разів його зарплата відрізняється від середньої **того ж року**. `mean()` дає 9 чисел, а потрібно 225 — по одному на рядок.

In [ ]:
year_mean = regions.groupby("рік")["зарплата"].transform("mean")
print("agg:      ", regions.groupby("рік")["зарплата"].mean().shape)
print("transform:", year_mean.shape)

In [ ]:
relative = regions.assign(до_середньої=(regions["зарплата"] / year_mean).round(2))
look(relative.query("рік == 2018").sort_values("до_середньої", ascending=False), "2018, відносно середньої", n=4)

`transform("mean")` повертає Series **тієї ж довжини**, що й таблиця (225), — у кожному рядку середнє його року. Тому ділення рядок-на-рядок працює. У 2018 році Київ — 1,69 від середньої по регіонах, Донецька — 1,21.

Правило: `agg` — одне значення на групу (коротша таблиця), `transform` — значення групи в кожному рядку (та сама довжина).

### Приклад 5.5. `rank` у групі: місце регіону щороку

In [ ]:
ranked = regions.assign(місце=regions.groupby("рік")["зарплата"].rank(ascending=False).astype(int))
places = ranked.pivot(index="регіон", columns="рік", values="місце")
print(places.loc[["м. Київ", "Донецька", "Луганська", "Тернопільська"], [2010, 2014, 2018]])

`groupby("рік")[...].rank(ascending=False)` — місце регіону серед 25 у кожному році (1 = найбільша зарплата), теж «рядок у рядок», як `transform`. Київ завжди перший, Донецька — друга, Тернопільська — остання. Луганська область з 5-го місця у 2010 році опускається на 17-те у 2018-му: після 2014 року зарплату там рахують лише по підконтрольній частині області.

---
## 6. `merge`: зарплата проти ціни хліба

Дві таблиці з різних джерел: зарплати (Держстат, рік) і ціни (WFP, місяць). `merge` — це SQL `JOIN`: рядки з'єднуються за **ключем**. Спершу зводимо обидві таблиці до спільного ключа — **року**.

Для ціни беремо офіційний WFP «National Average» житнього хліба (є лише за 2014–2017, одиниця — `Loaf`); для зарплати — рядок `Україна*` (без тимчасово окупованих територій — так рахує Держстат з 2010 року).

### Приклад 6.1. Звести ціни до року

In [ ]:
bread = (
    food.query("commodity == 'Bread (rye)' and admin1.isna()")
    .assign(рік=lambda d: d["date"].dt.year)
    .groupby("рік", as_index=False)
    .agg(ціна_хліба=("price", "mean"), місяців=("price", "size"))
)
look(bread, "хліб по роках", n=4)

In [ ]:
ukr_salary = salary.loc[salary["регіон"] == "Україна*", ["рік", "зарплата"]]
look(ukr_salary, "зарплата Україна*")

Хліб — 4 рядки (2014–2017; у 2014 році лише 10 місяців, бо ряд починається з березня, у 2017 — 11). Зарплата — 15 рядків (2000–2018), з них за 2000–2009 — `NaN`.

### Приклад 6.2. `merge(how="inner")` і похідний показник

In [ ]:
power = bread.merge(ukr_salary, on="рік", how="inner")
power["буханок"] = (power["зарплата"] / power["ціна_хліба"]).round(0)
look(power.round(2), "буханок на зарплату", n=4)

`inner` лишає лише роки, які є в **обох** таблицях: 2014–2017. Новий стовпець `буханок` — скільки буханок житнього хліба можна купити на середню зарплату.

**🔮 Прогноз:** Зарплата за 2014–2017 зросла вдвічі. Чи зросла вдвічі купівельна спроможність у «буханках»?

<details>
<summary>Відповідь</summary>

Ні: зарплата виросла з 3 480 до 7 104 грн (×2,04), а буханок — 619 → 458 → 511 → 595: у 2015 хліб подорожчав на 63%, а зарплата лише на 21%. Номінальне зростання ≠ реальне; щоб порівнювати роки, гроші треба «знецінювати» цінами.

</details>

### Приклад 6.3. Пастка 1: ключ різного типу

Ліна отримала від Тараса таблицю цін, де рік записаний **текстом** (`'2014'`) — так буває після `str.split` чи читання CSV без перетворення типів.

In [ ]:
bread_text_year = bread.assign(рік=bread["рік"].astype(str))
print("типи ключа:", bread_text_year["рік"].dtype, "проти", ukr_salary["рік"].dtype)
try:
    bread_text_year.merge(ukr_salary, on="рік")
except ValueError as error:
    print("ValueError →", str(error)[:110])

pandas не з'єднує текст `'2014'` з числом `2014` і прямо про це каже: `You are trying to merge on … and int64 columns` (текстовий тип — `str` у pandas 3, `object` у pandas 2). Добре, що падає: гірше, якби тихо дало 0 рядків. Лікування — привести ключ до одного типу: `.astype(int)`.

### Приклад 6.4. Пастка 2: ключі без пари — `how` та `indicator`

Роки в двох таблицях збігаються лише частково. Подивимось, скільки рядків дає кожен спосіб з'єднання.

In [ ]:
for how in ["inner", "left", "right", "outer"]:
    print(f"{how:6} → {len(bread.merge(ukr_salary, on='рік', how=how)):2} рядків")

In [ ]:
check = bread.merge(ukr_salary, on="рік", how="outer", indicator=True)
print(check["_merge"].value_counts())
print("лише в зарплатах:", check.loc[check["_merge"] == "right_only", "рік"].tolist())

- `inner` — 4 (лише спільні роки);
- `left` — 4 (усі роки хліба), `right` — 15 (усі роки зарплати, ціна там, де її немає, — `NaN`);
- `outer` — 15 (усе з обох боків).

`indicator=True` додає стовпець `_merge`: `both`, `left_only` чи `right_only`. Тут 4 роки — в обох, 11 — лише в зарплатах (2000, 2005–2013, 2018). Правило: після `merge` **порівнюй кількість рядків** до і після, а коли щось не сходиться — `indicator=True` покаже, які ключі загубились.

### Приклад 6.5. Пастка 3: «багато-до-багатьох» — рядки множаться

Тарас прислав ціни без назв ринків — лише `market_id` і область. Назви є в довіднику `wfp_markets_ukr.csv`. Ліна з'єднує за областю (`admin1`) — «вона ж є в обох таблицях».

In [ ]:
markets = pd.read_csv(MARKETS)
look(markets, "довідник ринків")
print("ринків на область (топ-3):", markets["admin1"].value_counts().head(3).to_dict())

In [ ]:
prices_lite = food[["date", "market_id", "admin1", "commodity", "price"]]
wrong = prices_lite.merge(markets[["admin1", "market"]], on="admin1")
print(f"до: {len(prices_lite):,} рядків → після merge за admin1: {len(wrong):,}")

Рядків стало в **10 разів більше**. У довіднику 268 ринків, і на одну область їх буває багато (Донецька — 51, Запорізька — 42). `merge` з'єднує кожну ціну з **кожним** ринком своєї області: «багато цін» × «багато ринків» = вибух рядків, і середні по такій таблиці — неправда.

Захист — параметр `validate`: ми очікуємо, що в довіднику ключ унікальний (`many_to_one`), і pandas перевірить це сам.

In [ ]:
try:
    prices_lite.merge(markets[["admin1", "market"]], on="admin1", validate="many_to_one")
except pd.errors.MergeError as error:
    print("MergeError →", error)

In [ ]:
right = prices_lite.merge(markets[["market_id", "market"]], on="market_id", how="left", validate="many_to_one")
look(right, "merge за market_id")
print("рядків стільки ж:", len(right) == len(prices_lite), "| без назви:", right["market"].isna().sum())

`validate="many_to_one"` впав одразу з поясненням, що ключ праворуч не унікальний. Правильний ключ — `market_id`: він унікальний у довіднику, кількість рядків не змінилась (112 259), кожна ціна отримала назву ринку, і жодної без пари.

### Приклад 6.6. Ключі різними мовами: довідник-місток

Тепер Ліна хоче порахувати «хліб на зарплату» **по областях** за 2018 рік. Зарплати — українською (`Вінницька`), ринки WFP — латиницею (`Vinnytsya`). Спільного ключа немає — його треба створити: словник «регіон → ринок». Київ і Київська область — **різні** ринки (`Kyiv city` і `Kyiv`), тому мапимо на ринок, а не на `admin1` (у WFP обидва мають `admin1 == "Kyiv"`).

In [ ]:
REGION_TO_MARKET = {
    "Автономна Республіка Крим": "Crimea", "Вінницька": "Vinnytsya", "Волинська": "Volyn",
    "Дніпропетровська": "Dnipropetrovsk", "Донецька": "Donetsk", "Житомирська": "Zhytomyr",
    "Закарпатська": "Zakarpattya", "Запорізька": "Zaporizhzhya", "Івано-Франківська": "Ivano-Frankivsk",
    "Київська": "Kyiv", "Кіровоградська": "Kirovohrad", "Луганська": "Luhansk", "Львівська": "Lviv",
    "Миколаївська": "Mykolaiv", "Одеська": "Odesa", "Полтавська": "Poltava", "Рівненська": "Rivne",
    "Сумська": "Sumy", "Тернопільська": "Ternopyl", "Харківська": "Kharkiv", "Херсонська": "Kherson",
    "Хмельницька": "Khmelnytskiy", "Черкаська": "Cherkasy", "Чернівецька": "Chernivtsi",
    "Чернігівська": "Chernihiv", "м. Київ": "Kyiv city", "м. Севастополь": "m. Sevastopol",
}

In [ ]:
salary2018 = (
    salary.query("рік == 2018 and not регіон.str.startswith('Україна')", engine="python")
    .assign(market=lambda d: d["регіон"].map(REGION_TO_MARKET))
)
december = pd.Timestamp("2018-12-15")
bread2018 = (
    food.query("commodity == 'Bread (first grade flour)' and date == @december")
    [["market", "price"]].rename(columns={"price": "ціна"})
)
look(salary2018, "зарплати 2018 з ключем market")
look(bread2018, "хліб, грудень 2018")
print("регіонів без ринку у словнику:", salary2018["market"].isna().sum())

In [ ]:
diagnosis = salary2018.merge(bread2018, on="market", how="outer", indicator=True)
print(diagnosis["_merge"].value_counts())
print(diagnosis.loc[diagnosis["_merge"] != "both", ["регіон", "market", "зарплата", "ціна", "_merge"]].to_string(index=False))

Зарплати — 27 регіонів, хліб — 25 ринків (WFP публікує регіональні ціни з грудня 2018 року; хліб з борошна першого ґатунку, ціна за кг). Словник покрив усі 27 регіонів. `outer` + `indicator` показує: 25 пар знайшлись, а Крим і Севастополь — `left_only`: для них немає ні зарплати 2018 року, ні ціни. Це очікувана втрата, а не помилка ключа: її ми свідомо приймаємо.

`query(..., engine="python")` потрібен через `.str.startswith` усередині рядка: стандартний рушій `query` методи `.str` не завжди підтримує.

### Задача 2. Скільки кілограмів хліба на середню зарплату — по областях

Склади таблицю `regional` зі стовпцями `регіон`, `зарплата`, `ціна`, `кг_хліба`:

- з'єднай `salary2018` і `bread2018` за `market` так, щоб лишились **лише пари** (25 рядків), і попроси pandas перевірити, що з обох боків ключ унікальний (`validate="one_to_one"`);
- `кг_хліба` = `зарплата / ціна`, округли до цілого;
- відсортуй від найбільшого `кг_хліба` до найменшого й скинь індекс (`reset_index(drop=True)`).

Перевірка — у наступній клітинці.

In [ ]:
# YOUR CODE HERE


In [ ]:
assert "regional" in globals(), "створи regional у клітинці задачі вище"
assert list(regional.columns) == ["регіон", "зарплата", "ціна", "кг_хліба"], "стовпці: регіон, зарплата, ціна, кг_хліба"
assert len(regional) == 25, f"лише пари: очікуємо 25 рядків, а є {len(regional)} — how='inner'?"
assert regional["кг_хліба"].notna().all()
assert regional["кг_хліба"].is_monotonic_decreasing, "відсортуй від більшого до меншого"
assert regional.loc[0, "регіон"] == "м. Київ" and regional.loc[0, "кг_хліба"] == 637, "кг_хліба = зарплата / ціна, округлено до цілого"
assert regional.loc[1, "регіон"] == "Одеська"
print(regional.head(3).to_string(index=False))
print("…")
print(regional.tail(3).to_string(index=False))
print("✅ Задача 2 розв'язана")

Київ — 637 кг хліба на середню зарплату. Друга — Одеська область, хоча зарплата там нижча, ніж у Донецькій чи Київській: хліб в Одесі найдешевший (13,32 грн/кг). Найменше — Житомирська (371 кг): зарплата середня, а хліб — один з найдорожчих. Купівельна сила залежить від **обох** чисел — саме тому ми їх і з'єднували.

---
## 7. Форма таблиці: `pivot`, `pivot_table`, `melt`

`climate-change_ukr.csv` — 48 показників Світового банку для України, кожен рядок = (показник, рік, значення). Це «довгий» формат. Для порівняння показників зручніша «широка» таблиця: рядок — рік, стовпець — показник. `pivot` — операція, обернена до `melt`.

### Приклад 7.1. `pivot`: з довгої — у широку

In [ ]:
wb_long = pd.read_csv(WORLD_BANK)
look(wb_long[["Year", "Indicator Code", "Value"]], "Світовий банк, довга")
wb = wb_long.pivot(index="Year", columns="Indicator Code", values="Value").sort_index()
look(wb.iloc[:, :4], "широка (перші 4 показники)")
print("розмір широкої:", wb.shape, "| показників у довгій:", wb_long["Indicator Code"].nunique())

1 405 рядків «показник-рік-значення» → таблиця 66 років (1960–2025) × 48 показників. Багато клітинок — `NaN`: різні показники Світовий банк рахує за різні роки (`AG.LND.ARBL.ZS` на початку ряду порожній).

### Приклад 7.2. Пік населення

In [ ]:
population = wb["SP.POP.TOTL"].dropna()
peak_year = int(population.idxmax())
print("пік:", peak_year, f"— {population.max() / 1e6:.2f} млн")
print("населення:", population.loc[[1991, 1993, 2000, 2022, 2024]].map("{:,.0f}".format).to_dict())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.5))
(population / 1e6).plot(ax=ax, color="#3b82c4", linewidth=2)
ax.axvline(peak_year, color="#888888", linestyle=":", label=f"пік {peak_year}")
ax.set_title("Населення України, млн (Світовий банк)"); ax.set_xlabel("Рік"); ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout(); plt.show()

`idxmax()` — мітка індексу (рік) з найбільшим значенням: 1993, 52,35 млн; 2024 — 37,86 млн. Далі населення лише зменшується, а 2022 року — різкий злам (за оцінкою Світового банку).

### Приклад 7.3. `pivot` падає на дублікатах → `pivot_table`

Ліна хоче таблицю «дата × товар» з цінами. Спроба `pivot`:

In [ ]:
basket_long = food.query("commodity in ['Sugar', 'Potatoes', 'Oil (sunflower)'] and admin1.notna()")
try:
    basket_long.pivot(index="date", columns="commodity", values="price")
except ValueError as error:
    print("ValueError →", error)

In [ ]:
basket_wide = basket_long.pivot_table(index="date", columns="commodity", values="price", aggfunc="mean").round(2)
look(basket_wide.tail(3), "дата × товар, середнє по ринках")
print(basket_wide.shape)

`pivot` не знає, що робити, коли на одну пару (дата, товар) припадає кілька значень — тут їх по одному на **кожен ринок**. `pivot_table` вирішує це агрегацією (`aggfunc="mean"` — середнє по ринках). Правило: `pivot` — коли пара «рядок × стовпець» унікальна (як у Світовому банку), `pivot_table` — коли треба ще й агрегувати.

### Приклад 7.4. `melt` назад: та сама інформація, інша форма

In [ ]:
back = wb.reset_index().melt(id_vars="Year", var_name="Indicator Code", value_name="Value").dropna(subset=["Value"])
look(back, "широка → знову довга")
print("рядків у довгій з файлу:", len(wb_long), "| після pivot + melt:", len(back))

`melt` дав 66 × 48 = 3 168 рядків, з них лише 1 405 зі значенням — рівно стільки, скільки в початковому файлі. Форма змінюється, інформація — ні: «широка» лише додає порожні клітинки там, де даних немає. Обирай форму під задачу: групування й графіки — довга, порівняння стовпців — широка.

---
## 8. Час: щоденні дані портів → місяці й роки

`ports` (розділ 1.6) — щоденні заходи суден і оцінки імпорту/експорту в тоннах по 16 портах, 2019–2026.

### Приклад 8.1. Пастка: дата з часовим поясом

In [ ]:
war = pd.Timestamp("2022-02-24")
try:
    ports[ports["date"] >= war]
except TypeError as error:
    print("TypeError →", error)

In [ ]:
WAR = pd.Timestamp("2022-02-24", tz="UTC")
after = ports[ports["date"] >= WAR]
print(f"рядків після {WAR.date()}: {len(after):,} з {len(ports):,}")

Дату з поясом (`UTC`) не можна порівнювати з датою без поясу — pandas не вгадує, яку годину мається на увазі. Рішення — створити межу теж з поясом: `pd.Timestamp("2022-02-24", tz="UTC")`. (Інший шлях — прибрати пояс з даних: `ports["date"].dt.tz_localize(None)`.)

### Приклад 8.2. Пастка: зріз за датами потребує відсортованого індексу

In [ ]:
by_date = ports.set_index("date")
try:
    by_date.loc["2022-02-20":"2022-02-27"]
except KeyError as error:
    print("KeyError →", str(error)[:100])

In [ ]:
by_date = by_date.sort_index()
week = by_date.loc["2022-02-20":"2022-02-27"]
look(week[["portname", "portcalls", "export"]], "тиждень навколо 24.02.2022")
print(week.groupby(week.index.date)["portcalls"].sum().to_dict())

Файл відсортований від нових дат до старих, і зріз `"2022-02-20":"2022-02-27"` на такому індексі pandas не виконує. Після `sort_index()` — працює: 8 днів × 16 портів = 128 рядків. Сума заходів суден за день видна прямо: 21–23 лютого — 22–25 на день, 24-го — 5, а 27-го — жодного.

### Приклад 8.3. `resample("YE")`: суми по роках

In [ ]:
yearly = by_date.resample("YE")[["portcalls", "export", "import"]].sum()
yearly.index = yearly.index.year
print((yearly[["export", "import"]] / 1e6).round(1).rename(columns=lambda c: c + ", млн т").to_string())
drop_pct = round(100 * (1 - yearly.loc[2022, "export"] / yearly.loc[2021, "export"]))
print(f"експорт 2022 проти 2021: −{drop_pct}%")

`resample` групує рядки за **періодами часу** на `DatetimeIndex`: `"YE"` — рік (мітка — кінець року), `"ME"` — місяць, `"W"` — тиждень. Індекс замінили на номер року для читабельності. Експорт 2022 року — на 63% менший, ніж 2021-го; 2023 — найгірший у даних; 2026 — лише до 27 березня.

### Приклад 8.4. `resample` і `groupby(dt.year)` — той самий результат?

In [ ]:
by_groupby = ports.groupby(ports["date"].dt.year)["export"].sum()
print("однаково:", np.allclose(by_groupby.to_numpy(), yearly["export"].to_numpy()))
monthly = by_date.resample("ME")["export"].sum() / 1e6
print("місяців:", len(monthly), "| перший:", monthly.index[0].date(), "| останній:", monthly.index[-1].date())

Для років — так. Але `resample` вміє будь-які періоди (місяці, тижні, квартали) і створює **порожні** періоди, якщо в даних є «дірка» (там буде 0 або `NaN`), а `groupby` по номеру місяця не знав би, що місяць пропущено. 87 місяців: від січня 2019 до березня 2026.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.5))
monthly.plot(ax=ax, color="#2e7d32", linewidth=1.8)
ax.axvline(WAR, color="#c62828", linestyle="--", label="24.02.2022")
ax.set_title("Експорт через порти України, млн т на місяць"); ax.set_xlabel(""); ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout(); plt.show()

**🔮 Прогноз:** 2023 рік — найгірший за експортом у даних. Чому 2024-й знову вдвічі більший?

<details>
<summary>Відповідь</summary>

Дані цього не пояснюють — лише показують. Контекст (морський коридор з осені 2023) — зовнішнє знання. Аналітик відділяє «що в даних» від «чому» (див. довідник: діагностична аналітика).

</details>

### Приклад 8.5. Групування + `resample`: роки по кожному порту

Сума по країні може приховувати, **звідки** взялися тонни. Розкладемо експорт по портах.

In [ ]:
per_port = by_date.groupby("portname")["export"].resample("YE").sum().reset_index()
per_port["рік"] = per_port["date"].dt.year
look(per_port, "порт × рік (довга)")
port_table = (per_port.pivot(index="portname", columns="рік", values="export") / 1e6).round(1)
print(port_table.loc[["Pivdennyi", "Chornomorsk", "Odesa", "Izmail", "Kerch"], [2021, 2023, 2024, 2025]])

`groupby("portname")[...].resample("YE")` — `resample` усередині кожної групи: 16 портів × 8 років = 128 рядків, далі `pivot` — у таблицю «порт × рік». Велика Одеса (Південний, Чорноморськ, Одеса) у 2024 році повернулась майже до довоєнних обсягів, Ізмаїл (Дунай) виріс у рази.

А от 2025 рік виглядає дивно: великі одеські порти — майже нуль, зате Керч (тимчасово окупований Крим) — понад 50 млн т. Загальна сума за 2025 рік (56,8 млн т) не насторожувала, а розклад по портах — так. Дані цього не пояснюють: це питання до джерела (оцінки за супутниковими сигналами суден можуть приписувати вантаж не тому порту). Висновок «експорт 2025 року пішов через Керч» робити **не можна** — це і є перевірка даних з різних боків.

### Приклад 8.6. «До/після»: маска за датою

In [ ]:
before_mask = by_date.index < WAR
daily_calls = by_date.groupby(level="date")["portcalls"].sum()
print(f"заходів суден на день: до — {daily_calls[daily_calls.index < WAR].mean():.1f}, "
      f"після — {daily_calls[daily_calls.index >= WAR].mean():.1f}")
print("днів до:", before_mask.sum() // 16, "| днів після:", (~before_mask).sum() // 16)

Спершу сума по всіх портах за **кожен день** (`groupby(level="date")` — групування за індексом), потім середнє окремо до і після 24.02.2022: близько 19,5 заходу суден на день до і близько 9 — після. Порівнюємо середні за день, а не суми: періоди різної довжини (1 150 і 1 493 дні).

### Приклад 8.7. Скільки днів порти «мовчали»

In [ ]:
silent = daily_calls[daily_calls == 0]
print("днів без жодного заходу судна:", len(silent))
print("перший:", silent.index.min().date(), "| останній:", silent.index.max().date())
print(silent.resample("ME").size()[lambda s: s > 0].rename(lambda d: d.strftime("%Y-%m")).to_dict())

Маска `daily_calls == 0` знаходить дні, коли жоден порт не прийняв судна; `resample("ME").size()` рахує такі дні по місяцях. Найбільше — березень і квітень 2022 року (14 з 27 днів). Поодинокі «нульові» дні у 2020 і 2024–2025 роках — радше прогалини в оцінках, ніж справжня зупинка всіх портів: такий день варто перевірити, перш ніж робити з нього висновок.

### Задача 3. Які порти виграли, а які втратили після 24.02.2022

Для кожного порту порахуй **середню кількість заходів суден на день** до і після 24.02.2022 (у `ports` — один рядок на день на порт, тож середнє `portcalls` по рядках порту — це і є «на день»):

- `calls` — DataFrame з індексом `portname` і стовпцями `до`, `після`, `зміна` (`після − до`), округленими до 2 знаків, відсортований за `зміна` від більшої до меншої;
- `winners` — **список** назв портів, у яких `зміна > 0.5`, у порядку сортування.

Підказка: стовпець `період` зі значеннями `"до"`/`"після"` (через `np.where` з маскою `ports["date"] < WAR`) → `groupby(["portname", "період"])` → `unstack()`. Не забудь, що `date` — з часовим поясом.

Перевірка — у наступній клітинці.

In [ ]:
# YOUR CODE HERE


In [ ]:
assert "calls" in globals(), "створи calls і winners у клітинці задачі вище"
assert list(calls.columns) == ["до", "після", "зміна"], "стовпці: до, після, зміна"
assert len(calls) == 16, f"очікуємо 16 портів, а є {len(calls)}"
assert calls["зміна"].is_monotonic_decreasing, "відсортуй за зміною від більшої до меншої"
assert calls.loc["Odesa", "до"] == 3.2, "Одеса до війни — 3.2 заходи на день: перевір межу дати (tz='UTC')"
assert winners == ["Kerch", "Izmail", "Reni"], winners
print(calls.head(4))
print("…")
print(calls.tail(3))
print("✅ Задача 3 розв'язана")

Ізмаїл і Рені — дунайські порти: коли морські порти закрились, частина вантажів пішла Дунаєм. Керч — тимчасово окупований порт; зростання там — окрема історія, і, як ми бачили в прикладі 8.5, ці оцінки варто перевіряти.

---
## 9. Ціни у світі: великий файл без зайвого

`wfp_food_prices_global_2026.csv` — 32 МБ, 245 тис. рядків, 71 країна. Читаємо лише потрібні стовпці (`usecols`) — у рази менше пам'яті й часу.

### Приклад 9.1. `usecols`: скільки пам'яті економимо

In [ ]:
COLUMNS = ["countryiso3", "commodity", "unit", "pricetype", "usdprice"]

In [ ]:
full = pd.read_csv(WORLD, low_memory=False)
world = pd.read_csv(WORLD, usecols=COLUMNS)
full_mb = full.memory_usage(deep=True).sum() / 1e6
world_mb = world.memory_usage(deep=True).sum() / 1e6
print(f"усі {full.shape[1]} стовпців: {full_mb:.0f} МБ | {len(COLUMNS)} стовпців: {world_mb:.0f} МБ — у {full_mb / world_mb:.1f} раза менше")
look(world, "світ, лише потрібні стовпці")
del full

Повний файл у пам'яті займає в кілька разів більше, ніж потрібні 5 стовпців (точні мегабайти залежать від версії pandas: у pandas 3 текст зберігається компактніше). `del full` звільняє пам'ять — таблиця більше не потрібна.

### Приклад 9.2. Медіанна ціна цукру по країнах і місце України

In [ ]:
sugar = (
    world.query("commodity == 'Sugar' and unit == 'KG' and pricetype == 'Retail'")
    .groupby("countryiso3")["usdprice"].median()
    .sort_values()
)
ukr_rank = int(sugar.index.get_loc("UKR")) + 1
print(f"країн: {len(sugar)} | Україна: {ukr_rank} місце, ${sugar['UKR']:.2f}/кг | медіана країн: ${sugar.median():.2f}")
print("найдешевше:", sugar.head(3).round(2).to_dict())
print("найдорожче:", sugar.tail(3).round(2).to_dict())

Фільтр (лише цукор, кілограм, роздріб — не змішуємо одиниці!), групування за країною, **медіана** (стійка до поодиноких дуже дорогих ринків), сортування. `index.get_loc("UKR")` — позиція України в упорядкованому індексі (з нуля, тому `+ 1`). Україна — 4-те місце з 40 за дешевизною: $0,66/кг при медіані серед країн $1,06. Ціни в доларах порівнюють «вартість у світі», а не доступність для людей: для неї потрібні доходи — як «буханки на зарплату» в розділі 6.

---
## Самоперевірка

1. Як зрозуміти, що файл у cp1251, і що буде, якщо прочитати його як UTF-8?
2. Які параметри `read_csv` потрібні для файлу з `;`, дробовою комою й датами `15.03.2024`?
3. Навіщо `melt` для таблиці зарплат і `pivot` для таблиці Світового банку — це протилежні операції? Коли `pivot` падає?
4. Чим `groupby(...).mean()` відрізняється від `groupby(...).transform("mean")`?
5. Чому `salary[salary["рік"] >= 2016 & salary["зарплата"] > 8000]` падає?
6. Коли `merge` втрачає рядки, коли — множить їх, і як це помітити?
7. Що робить `resample("YE")` і чим він відрізняється від `groupby(date.dt.year)`?
8. Чому `ports["date"] >= pd.Timestamp("2022-02-24")` падає?
9. Чому «буханок на зарплату» — кращий показник, ніж сама зарплата?

<details>
<summary>Відповіді</summary>

1. Байти `\xca\xe2…` не складаються в UTF-8; `pd.read_csv` без `encoding` впаде з `UnicodeDecodeError` або покаже «кракозябри».
2. `sep=";"`, `decimal=","`, `parse_dates=["дата"]` з `date_format="%d.%m.%Y"` (або `pd.to_datetime(..., dayfirst=True)` після читання).
3. Так: `melt` — «широка» → «довга», `pivot` — навпаки. `pivot` падає, якщо пара (рядок, стовпець) трапляється кілька разів, — тоді `pivot_table` з `aggfunc`.
4. `mean()` — одне значення на групу (коротша таблиця); `transform("mean")` — значення групи в кожному рядку (та сама довжина), зручно для «частки від групи».
5. `&` виконується раніше за `>=` і `>`; кожну умову треба взяти в дужки.
6. Втрачає — коли ключа немає з одного боку (`how="inner"`), множить — коли ключ не унікальний з обох боків («багато-до-багатьох»). Порівнюй `len` до і після, `indicator=True` показує ключі без пари, `validate="many_to_one"` перевіряє унікальність.
7. Групує за періодами часу на `DatetimeIndex`; для років результат той самий, але `resample` вміє місяці, тижні, квартали й створює порожні періоди.
8. Дати в файлі — з часовим поясом UTC, а межа — без нього; треба `pd.Timestamp("2022-02-24", tz="UTC")`.
9. Він прибирає інфляцію: показує, що можна **купити**, а не скільки гривень.

</details>

## Далі

- Книга курсу: [Бонус. Pandas: аналіз даних, графіки і Dash](https://nikoriakviktor.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m3/bonus_pandas/).
- Довідник: [аналіз даних — патерни й мислення](https://nikoriakviktor.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m3/pandas/data_analytics/).
- Дашборд: `cd dash_API && python app.py` → `http://localhost:8055`.